# Soal 3 · WORKER: Spark DataFrame & SQL pada data milik sendiri
**Big Data** · dijalankan di **PC Worker masing-masing** (Worker 1 sampai 4), lewat JupyterLab dari dashboard PC itu sendiri.

Pola sama dengan materi *Kafka_Cuaca_Cluster*: notebook ini **menunggu tugas dari Master lewat Kafka**, mengambil data
bagiannya, mengolahnya dengan **Spark lokal `local[2]`** (DataFrame + SQL), lalu mengirim **ringkasan** ke Master.

| Tugas | Isi |
|---|---|
| `olah` | ambil bagian file snapshot milik satu partisi Kafka → skema eksplisit → data rusak → view `edits` → 7 kueri SQL (W1–W7) |
| `performa` | `explain()` agregasi, percobaan `shuffle.partitions` (E2), join sort-merge vs broadcast (E5), metrik stage dari Spark UI |

**Data bagian Worker = satu partisi Kafka.** Producer memakai key `wiki:title`, jadi satu halaman selalu berada di satu partisi;
kueri per halaman (W5, W6, W7) sudah tuntas di Worker.

**Syarat:** folder `C:\BigData\data\snapshot_wiki\` berisi `wiki_snapshot_uji.jsonl` (sama seperti Soal 2), `confluent_kafka` dan
`pyspark` terpasang di venv. **Isi Sel 1:** `NOMOR_WORKER` (1–4) dan `MASTER_IP`. **Urutan:** Sel 1 → 2 → 3 → 4 → 5 → 6
(Sel 6 terus berjalan sampai di-Interrupt).

## Sel 1. Konfigurasi, Kafka, dan pelapor

In [1]:
# ===== SEL 1: Konfigurasi, Kafka, dan fungsi pelapor (PC Worker) =====
import os, sys, io, json, time, socket, contextlib, urllib.request
from pathlib import Path
from confluent_kafka import Producer, Consumer, KafkaError, KafkaException

# ---------------- ISI SESUAI PC INI ----------------
KELOMPOK     = "wiki3"                       # HARUS sama dengan Master
NOMOR_WORKER = 4                             # 1 ... 4
MASTER_IP    = "192.168.1.15"                # IP PC Master (broker Kafka)
DATA_DIR     = Path("C:/BigData/data/snapshot_wiki")
# ---------------------------------------------------

BOOTSTRAP    = f"{MASTER_IP}:9092"
TOPIC_TUGAS  = f"{KELOMPOK}-tugas"
TOPIC_HASIL  = f"{KELOMPOK}-hasil"
TOPIC_STATUS = f"{KELOMPOK}-status"
GROUP_ID     = f"{KELOMPOK}-pekerja"          # semua Worker = satu consumer group
HOST         = f"{socket.gethostname()}-w{NOMOR_WORKER}"
UKURAN_BATCH = 500                           # baris ringkasan per pesan Kafka
KERJA_DIR    = Path("soal3_kerja"); KERJA_DIR.mkdir(exist_ok=True)   # lapisan raw lokal
RUN = "-"

produser = Producer({"bootstrap.servers": BOOTSTRAP, "acks": "all", "enable.idempotence": True,
                     "compression.type": "gzip", "linger.ms": 50})
galat_kirim = []
def _hasil_kirim(err, msg):
    if err is not None:
        galat_kirim.append(str(err))

def kirim(topic, key, obj):
    produser.produce(topic, key=key, value=json.dumps(obj, default=str).encode("utf-8"),
                     on_delivery=_hasil_kirim)
    produser.poll(0)

def lapor(tugas, status, **tambahan):
    kirim(TOPIC_STATUS, HOST, {"run": RUN, "worker": HOST, "tugas": tugas["tugas"],
                               "status": status, "waktu": time.time(), **tambahan})

def lapor_siap(consumer, partisi):
    kirim(TOPIC_STATUS, HOST, {"run": "-", "worker": HOST, "status": "SIAP", "waktu": time.time(),
                               "partisi": sorted(p.partition for p in partisi)})
    produser.flush()
    print("Partisi tugas yang dipegang:", sorted(p.partition for p in partisi))

def kirim_tabel(tugas, tabel, baris):
    """Kirim satu tabel ringkasan dalam potongan UKURAN_BATCH baris. Mengembalikan jumlah potongan."""
    potongan = [baris[i:i + UKURAN_BATCH] for i in range(0, len(baris), UKURAN_BATCH)] or [[]]
    for no, isi in enumerate(potongan):
        kirim(TOPIC_HASIL, f"{RUN}-{tugas['tugas']}-{tabel}-{no}",
              {"run": RUN, "tugas": tugas["tugas"], "worker": HOST, "tabel": tabel,
               "no": no, "dari": len(potongan), "baris": isi})
    return len(potongan)

meta = produser.list_topics(timeout=5)
print("Terhubung ke Kafka. Topic kelompok:", sorted(t for t in meta.topics if t.startswith(KELOMPOK)))
print("Nama Worker :", HOST)
print("File uji    :", DATA_DIR / "wiki_snapshot_uji.jsonl", "->",
      "ADA" if (DATA_DIR / "wiki_snapshot_uji.jsonl").exists() else "TIDAK ADA (salin dulu seperti Soal 2)")

Terhubung ke Kafka. Topic kelompok: ['wiki3-hasil', 'wiki3-status', 'wiki3-tugas']
Nama Worker : NthaanSnn-w4
File uji    : C:\BigData\data\snapshot_wiki\wiki_snapshot_uji.jsonl -> ADA


## Sel 2. Spark lokal, skema eksplisit, kueri SQL, dan alat ukur
Skema ditulis **eksplisit** (bukan `inferSchema`), 16 field sesuai pesan producer + kolom `_corrupt_record` untuk baris rusak.
Nama di soal `length.new` / `length.old` (struktur bersarang Wikimedia) = `length_new` / `length_old` di pesan producer kita.

In [2]:
# ===== SEL 2: Spark lokal (DataFrame & SQL), skema, dan kueri =====
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import (StructType, StructField, StringType, LongType, IntegerType,
                               BooleanType, DoubleType)

spark = (SparkSession.builder
         .appName(f"{KELOMPOK}-worker{NOMOR_WORKER}")
         .master("local[2]")                                  # Worker mengolah datanya sendiri
         .config("spark.driver.memory", "1g")
         .config("spark.driver.host", "127.0.0.1")
         .config("spark.driver.bindAddress", "127.0.0.1")
         .config("spark.sql.shuffle.partitions", "8")         # data kecil: 8 cukup (bawaan 200)
         .config("spark.sql.session.timeZone", "Asia/Jakarta")
         .config("spark.pyspark.python", sys.executable)
         .config("spark.pyspark.driver.python", sys.executable)
         .getOrCreate())
sc = spark.sparkContext
sc.setLogLevel("ERROR")
print("Spark", spark.version, "| master:", sc.master, "| UI:", sc.uiWebUrl)

SKEMA = StructType([
    StructField("id",          LongType(),    True),
    StructField("type",        StringType(),  True),
    StructField("wiki",        StringType(),  True),
    StructField("server_name", StringType(),  True),
    StructField("title",       StringType(),  True),
    StructField("namespace",   IntegerType(), True),
    StructField("user",        StringType(),  True),
    StructField("is_anon",     BooleanType(), True),
    StructField("anon_kind",   StringType(),  True),
    StructField("bot",         BooleanType(), True),
    StructField("minor",       BooleanType(), True),
    StructField("timestamp",   LongType(),    True),   # detik epoch
    StructField("length_old",  LongType(),    True),   # null untuk type = new
    StructField("length_new",  LongType(),    True),
    StructField("comment",     StringType(),  True),
    StructField("ingest_ts",   DoubleType(),  True),
    StructField("_corrupt_record", StringType(), True),
])

# Field wajib sama dengan aturan Soal 2a: id, timestamp ada; wiki, type, title berupa teks tidak kosong
WAJIB = (F.col("id").isNotNull() & F.col("timestamp").isNotNull()
         & (F.length("wiki") > 0) & (F.length("type") > 0) & (F.length("title") > 0))

# Kueri SQL Worker (view: edits, jendela). "edit" = type 'edit' seperti Soal 2.
SQL_JENDELA = """
SELECT wiki, title, window(ts, '5 minutes') AS w,
       COUNT(*)                                   AS n_edit,
       SUM(CASE WHEN bot THEN 1 ELSE 0 END)       AS n_bot,
       SUM(CASE WHEN NOT bot THEN 1 ELSE 0 END)   AS n_manusia,
       COUNT(DISTINCT user)                       AS editor_unik
FROM edits
WHERE type = 'edit'
GROUP BY wiki, title, window(ts, '5 minutes')
"""
SQL_WORKER = {
"W1_per_menit": """
SELECT wiki, date_format(ts, 'yyyy-MM-dd HH:mm') AS menit, COUNT(*) AS n
FROM edits WHERE type = 'edit'
GROUP BY wiki, date_format(ts, 'yyyy-MM-dd HH:mm')""",
"W2_bot_wiki": """
SELECT wiki, bot, COUNT(*) AS n
FROM edits WHERE type = 'edit'
GROUP BY wiki, bot""",
"W3_selisih_ringkas": """
SELECT type,
       COUNT(selisih_byte)                         AS n,
       SUM(ABS(selisih_byte))                      AS jumlah_abs,
       ROUND(AVG(ABS(selisih_byte)), 2)            AS rata_abs,
       percentile_approx(ABS(selisih_byte), 0.5)   AS median_approx
FROM edits GROUP BY type""",
"W4_selisih_hist": """
SELECT type, ABS(selisih_byte) AS nilai_abs, COUNT(*) AS frekuensi
FROM edits WHERE selisih_byte IS NOT NULL
GROUP BY type, ABS(selisih_byte)""",
"W5_idwiki_halaman": """
SELECT title, COUNT(DISTINCT user) AS editor_unik, COUNT(*) AS n_edit,
       SUM(CASE WHEN bot THEN 1 ELSE 0 END) AS n_bot
FROM edits WHERE wiki = 'idwiki'
GROUP BY title""",
"W6_jendela_hist": """
SELECT 'semua' AS kelompok, n_edit AS n, COUNT(*) AS frekuensi FROM jendela GROUP BY n_edit
UNION ALL
SELECT 'manusia' AS kelompok, n_manusia AS n, COUNT(*) AS frekuensi FROM jendela
WHERE n_manusia > 0 GROUP BY n_manusia""",
"W7_kandidat": """
SELECT wiki, title, date_format(w.start, 'yyyy-MM-dd HH:mm') AS mulai,
       n_edit, n_bot, n_manusia, editor_unik
FROM jendela WHERE n_edit >= 2""",
}

def ukur(fn, ulang=3):
    """Jalankan fn beberapa kali, kembalikan waktu terbaik (detik)."""
    terbaik = None
    for _ in range(ulang):
        t0 = time.time(); fn(); d = time.time() - t0
        terbaik = d if terbaik is None else min(terbaik, d)
    return round(terbaik, 3)

def rencana(df, mode="simple"):
    """Tangkap keluaran df.explain() sebagai teks."""
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        df.explain(mode=mode)
    return buf.getvalue()

def jalankan_penuh(df):
    """Paksa seluruh perhitungan tanpa mengumpulkan hasil ke driver."""
    df.write.format("noop").mode("overwrite").save()

_TANPA_PROXY = urllib.request.build_opener(urllib.request.ProxyHandler({}))   # UI lokal, jangan lewat proxy

def api(path):
    url = f"{sc.uiWebUrl}/api/v1/applications/{sc.applicationId}/{path}"
    with _TANPA_PROXY.open(url, timeout=10) as r:
        return json.load(r)

def daftar_job():
    """Semua jobId dari status tracker Spark (Python API, tanpa HTTP)."""
    return set(sc.statusTracker().getJobIdsForGroup())      # tanpa argumen = job tanpa job group

def jalankan_tercatat(fn):
    """Jalankan fn, kembalikan jobId yang BARU muncul (dibandingkan sebelum dan sesudah)."""
    sebelum = daftar_job()
    fn()
    baru = sorted(daftar_job() - sebelum)
    if not baru:
        print("  PERINGATAN: tidak ada job baru tercatat di status tracker")
    return baru

def _rest_stage(sid):
    """Shuffle Write/Read dari REST API Spark UI; None bila UI tidak terjangkau (tidak menggagalkan tugas)."""
    try:
        return api(f"stages/{sid}")[0]
    except Exception:
        return None

def metrik_job(job_ids):
    """Stage dan task dari status tracker; Shuffle Write/Read dari REST API bila terjangkau."""
    st_tr, baris = sc.statusTracker(), []
    for jid in job_ids:
        job = st_tr.getJobInfo(jid)
        if job is None:
            continue
        for sid in sorted(job.stageIds):
            info = st_tr.getStageInfo(sid)          # None = stage tidak pernah dijalankan (SKIPPED)
            rest = _rest_stage(sid) if info is not None else None
            baris.append({"job": jid, "stage": sid,
                          "status": "SKIPPED" if info is None else (rest or {}).get("status", "COMPLETE"),
                          "nama": "" if info is None else str(info.name)[:40],
                          "task": 0 if info is None else info.numTasks,
                          "shuffle_write": (rest or {}).get("shuffleWriteBytes"),
                          "shuffle_read": (rest or {}).get("shuffleReadBytes"),
                          "run_ms": (rest or {}).get("executorRunTime")})
    return baris

print("Skema, kueri, dan alat ukur siap.")

Spark 4.1.3 | master: local[2] | UI: http://127.0.0.1:4040
Skema, kueri, dan alat ukur siap.


## Sel 3. Ingestion: mengambil bagian data milik partisi ini
Padanan "unduh data kota" di materi. Rentang baris dikirim Master di dalam tugas; rentang itu sudah memuat baris rusak sisipan
yang jatuh di bagian ini. **Baris kosong dibuang di sini** dan dihitung, karena pembaca JSON Spark melewati baris kosong tanpa menandainya.

In [3]:
# ===== SEL 3: Ingestion - ambil rentang baris dari file uji =====
def ingest(tugas):
    sumber = DATA_DIR / tugas["file"]
    mulai, akhir = tugas["baris_mulai"], tugas["baris_akhir"]       # indeks 0, akhir eksklusif
    tujuan = KERJA_DIR / f"bagian_p{tugas['partisi_kafka']}.jsonl"
    t0, dibaca, kosong, ditulis = time.time(), 0, 0, 0
    with open(sumber, "rb") as src, open(tujuan, "wb") as dst:
        for i, baris in enumerate(src):
            if i < mulai:
                continue
            if i >= akhir:
                break
            dibaca += 1
            if not baris.strip():
                kosong += 1
                continue
            dst.write(baris if baris.endswith(b"\n") else baris + b"\n")
            ditulis += 1
    if dibaca != akhir - mulai:
        raise RuntimeError(f"hanya {dibaca} dari {akhir - mulai} baris terbaca; file berbeda dengan milik Master?")
    return tujuan, {"baris_file": dibaca, "kosong": kosong, "ditulis": ditulis,
                    "detik_ingest": round(time.time() - t0, 2)}

print("Fungsi ingest siap.")

Fungsi ingest siap.


## Sel 4. Tugas `olah`: DataFrame + SQL (Soal 3a, ringkasan untuk 3b dan 3c)
Baca dengan skema → pisahkan baris rusak (`_corrupt_record`) dan baris tanpa field wajib → kolom `ts` (tipe timestamp) dan
`selisih_byte` → view `edits` → view `jendela` (5 menit per halaman) → kueri W1–W7.

In [4]:
# ===== SEL 4: Pengolahan (Soal 3a) dan ringkasan =====
def muat(path):
    """Soal 3a: DataFrame dengan skema eksplisit + kolom turunan. Mengembalikan (mentah, edits)."""
    mentah = (spark.read.schema(SKEMA)
              .option("mode", "PERMISSIVE")
              .option("columnNameOfCorruptRecord", "_corrupt_record")
              .json(Path(path).resolve().as_uri()))
    edits = (mentah.filter(F.col("_corrupt_record").isNull() & F.coalesce(WAJIB, F.lit(False)))
                   .drop("_corrupt_record")
                   .withColumn("ts", F.timestamp_seconds("timestamp"))
                   .withColumn("selisih_byte", F.col("length_new") - F.coalesce(F.col("length_old"), F.lit(0))))
    return mentah, edits

def olah(tugas):
    path, info = ingest(tugas)
    print(f"  ingest p{tugas['partisi_kafka']}: {info['baris_file']:,} baris file, {info['kosong']} kosong dibuang, "
          f"{info['detik_ingest']} dtk")
    lapor(tugas, "MEMPROSES", tahap="3a skema")
    t0 = time.time()
    mentah, edits = muat(path)
    mentah.cache()                      # wajib sebelum menyaring _corrupt_record saja (aturan Spark)
    n_mentah = mentah.count()
    n_corrupt = mentah.filter(F.col("_corrupt_record").isNotNull()).count()
    n_wajib = mentah.filter(F.col("_corrupt_record").isNull() & ~F.coalesce(WAJIB, F.lit(False))).count()
    edits.cache()
    edits.createOrReplaceTempView("edits")
    n_valid = edits.count()
    per_type = {r["type"]: r["n"] for r in spark.sql("SELECT type, COUNT(*) AS n FROM edits GROUP BY type").collect()}
    rentang = spark.sql("SELECT date_format(MIN(ts), 'yyyy-MM-dd HH:mm:ss') AS a, "
                        "date_format(MAX(ts), 'yyyy-MM-dd HH:mm:ss') AS b FROM edits").first()
    contoh = edits.select("id", "wiki", "type", "title", "ts", "length_old", "length_new", "selisih_byte").limit(3).toPandas()

    jendela = spark.sql(SQL_JENDELA).cache()
    jendela.createOrReplaceTempView("jendela")

    lapor(tugas, "MEMPROSES", tahap="SQL W1-W7")
    tabel = {}
    for nama, q in SQL_WORKER.items():
        tabel[nama] = [r.asDict() for r in spark.sql(q).collect()]
        print(f"  {nama:<20} {len(tabel[nama]):>8,} baris")
    detik = round(time.time() - t0, 2)

    tabel["W0_statistik"] = [{
        "partisi_kafka": tugas["partisi_kafka"], **info,
        "n_mentah_spark": n_mentah, "rusak_corrupt": n_corrupt, "rusak_field_wajib": n_wajib,
        "n_valid": n_valid, "per_type": per_type, "ts_min": rentang["a"], "ts_max": rentang["b"],
        "harap_valid": tugas["harap_valid"], "harap_rusak": tugas["harap_rusak"],
        "detik_proses": detik, "zona_waktu": spark.conf.get("spark.sql.session.timeZone"),
        "skema": SKEMA.simpleString(), "contoh": contoh.to_dict("records"),
    }]
    print(f"  valid {n_valid:,} (harap {tugas['harap_valid']:,}) | rusak: kosong {info['kosong']}, "
          f"corrupt {n_corrupt}, field_wajib {n_wajib} (harap total {sum(tugas['harap_rusak'].values())})")
    print(contoh.to_string(index=False))
    for d in (jendela, edits, mentah):
        d.unpersist()
    return tabel, {"baris": info["baris_file"], "detik_proses": detik}

print("Fungsi olah siap.")

Fungsi olah siap.


## Sel 5. Tugas `performa`: `explain()`, `shuffle.partitions`, dan broadcast join (Soal 3d)
| Kode | Percobaan | Catatan |
|---|---|---|
| P1 | `explain("formatted")` agregasi (ii) **sebelum** dan **sesudah** action | `maxPartitionBytes = 8m` agar stage pertama > 1 task |
| E2 | `shuffle.partitions` 200 / 8 / 2, **AQE dimatikan** | agregasi per (wiki, bot, salinan) pada data diperbesar |
| E5 | join `wiki_ref` (wiki → bahasa): sort-merge vs broadcast | `autoBroadcastJoinThreshold = -1` dan AQE dimatikan untuk sort-merge |

Waktu diukur dengan `ukur()` (terbaik dari 3) dan `jalankan_penuh()` (`noop`). Setiap varian dijalankan sekali lagi; job yang baru muncul di *status tracker* Spark
(dibandingkan sebelum dan sesudah) dipakai untuk mengambil jumlah stage/task (status tracker) dan Shuffle Write/Read (REST API Spark UI, bila terjangkau).

In [5]:
# ===== SEL 5: Percobaan performa (Soal 3d) =====
def performa(tugas):
    hasil_waktu, hasil_stage, hasil_rencana = [], [], []
    awal = {k: spark.conf.get(k) for k in ("spark.sql.adaptive.enabled", "spark.sql.shuffle.partitions",
                                          "spark.sql.autoBroadcastJoinThreshold", "spark.sql.files.maxPartitionBytes")}

    def catat(eks, var, detik, job_ids, cat=""):
        stg = metrik_job(job_ids)
        for s in stg:
            hasil_stage.append({"eksperimen": eks, "varian": str(var), **s})
        aktif = [s for s in stg if s["status"] != "SKIPPED"]
        hasil_waktu.append({"eksperimen": eks, "varian": str(var), "detik": detik, "catatan": cat,
                            "n_stage": len(aktif), "n_task": sum(s["task"] for s in aktif),
                            "shuffle_write": sum(s["shuffle_write"] or 0 for s in aktif),
                            "shuffle_read": sum(s["shuffle_read"] or 0 for s in aktif),
                            "rest_ui": all(s["shuffle_write"] is not None for s in aktif)})
        print(f"  {eks:<22} {str(var):<12} {detik:>7.3f} dtk | stage {len(aktif)} task "
              f"{sum(s['task'] for s in aktif):>4} | SW {hasil_waktu[-1]['shuffle_write']:>9,} B {cat}")

    path, _ = ingest(tugas)
    spark.conf.set("spark.sql.files.maxPartitionBytes", tugas["max_partition_bytes"])
    _, edits = muat(path)
    n_part = edits.rdd.getNumPartitions()
    print(f"  partisi input setelah maxPartitionBytes={tugas['max_partition_bytes']}: {n_part}")

    # P1: explain agregasi (ii) sebelum dan sesudah action
    lapor(tugas, "MEMPROSES", tahap="P1 explain")
    q2 = edits.filter(F.col("type") == "edit").groupBy("wiki", "bot").count()
    hasil_rencana.append({"nama": "P1 sebelum action (AQE aktif)", "teks": rencana(q2, "formatted")})
    jobs = jalankan_tercatat(lambda: q2.collect())
    hasil_rencana.append({"nama": "P1 sesudah action (AQE aktif)", "teks": rencana(q2, "formatted")})
    catat("P1 explain", f"input {n_part} partisi", 0.0, jobs)

    # Data diperbesar khusus percobaan waktu (kolom seperlunya saja agar muat di memori)
    besar = (edits.select("wiki", "bot", "type")
                  .withColumn("salinan", F.explode(F.array(*[F.lit(i) for i in range(tugas["faktor"])])))
                  .cache())
    n_besar = besar.count()
    print(f"  data diperbesar x{tugas['faktor']}: {n_besar:,} baris (di-cache)")
    agregat = lambda d: d.groupBy("wiki", "bot", "salinan").count()

    # E2: jumlah partisi shuffle, AQE dimatikan agar pengaruhnya terlihat
    lapor(tugas, "MEMPROSES", tahap="E2 shuffle")
    spark.conf.set("spark.sql.adaptive.enabled", "false")
    for n in (200, 8, 2):
        spark.conf.set("spark.sql.shuffle.partitions", str(n))
        detik = ukur(lambda: jalankan_penuh(agregat(besar)))
        jobs = jalankan_tercatat(lambda: jalankan_penuh(agregat(besar)))
        catat("E2 shuffle partitions", n, detik, jobs)
    hasil_rencana.append({"nama": "E2 shuffle.partitions=2", "teks": rencana(agregat(besar))})
    spark.conf.set("spark.sql.shuffle.partitions", awal["spark.sql.shuffle.partitions"])

    # E5: join dengan tabel referensi wiki -> bahasa
    lapor(tugas, "MEMPROSES", tahap="E5 join")
    ref = spark.createDataFrame(tugas["wiki_ref"], "wiki string, kode string, bahasa string")
    gabung = lambda r: (besar.join(r, "wiki", "left")
                             .withColumn("bahasa", F.coalesce("bahasa", F.lit("lainnya")))
                             .groupBy("bahasa").count())
    spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")
    j_sm = gabung(ref)
    teks = rencana(j_sm)
    detik = ukur(lambda: jalankan_penuh(j_sm))
    jobs = jalankan_tercatat(lambda: jalankan_penuh(j_sm))
    catat("E5 join", "sort-merge", detik, jobs, "SortMergeJoin" if "SortMergeJoin" in teks else "?")
    hasil_rencana.append({"nama": "E5 sort-merge", "teks": teks})
    spark.conf.set("spark.sql.autoBroadcastJoinThreshold", awal["spark.sql.autoBroadcastJoinThreshold"])
    j_bc = gabung(F.broadcast(ref))
    teks = rencana(j_bc)
    detik = ukur(lambda: jalankan_penuh(j_bc))
    jobs = jalankan_tercatat(lambda: jalankan_penuh(j_bc))
    catat("E5 join", "broadcast", detik, jobs, "BroadcastHashJoin" if "BroadcastHashJoin" in teks else "?")
    hasil_rencana.append({"nama": "E5 broadcast", "teks": teks})
    per_bahasa = [r.asDict() for r in j_bc.orderBy(F.desc("count")).collect()]

    besar.unpersist()
    for k, v in awal.items():
        spark.conf.set(k, v)
    return ({"P_waktu": hasil_waktu, "P_stage": hasil_stage, "P_rencana": hasil_rencana,
             "P_bahasa": [{"bahasa": r["bahasa"], "n": r["count"]} for r in per_bahasa]},
            {"baris": n_besar, "detik_proses": 0})

print("Fungsi performa siap.")

Fungsi performa siap.


## Sel 6. Worker mengerjakan tugas dari Kafka
Sel ini **terus berjalan**. Hentikan dengan Interrupt setelah Master menyatakan semua tugas selesai.

In [ ]:
# ===== SEL 6: Loop Worker =====
def kerjakan(tugas):
    global RUN
    RUN = tugas["run"]
    lapor(tugas, "MEMPROSES", tahap="mulai")
    produser.flush()
    t0 = time.time()
    try:
        tabel, info = olah(tugas) if tugas["jenis"] == "olah" else performa(tugas)
        lapor(tugas, "MENGIRIM")
        galat_kirim.clear()
        n_batch = sum(kirim_tabel(tugas, nama, baris) for nama, baris in tabel.items())
        sisa = produser.flush(120)
        if sisa > 0 or galat_kirim:
            raise RuntimeError(f"hasil belum terkirim (sisa={sisa}, galat={galat_kirim[:1]})")
        info["detik_total"] = round(time.time() - t0, 2)
        lapor(tugas, "SELESAI", jumlah=n_batch, **info)     # jumlah = janji; Master mencocokkannya
        produser.flush()
        print(f"Tugas {tugas['tugas']} ({tugas['jenis']}) selesai: {n_batch} potongan terkirim, {info['detik_total']} dtk")
    except Exception as e:
        lapor(tugas, "GAGAL", galat=str(e)[:200])
        produser.flush()
        print(f"Tugas {tugas['tugas']} GAGAL:", e)

konsumen = Consumer({
    "bootstrap.servers": BOOTSTRAP,
    "group.id": GROUP_ID,
    "auto.offset.reset": "earliest",
    "enable.auto.commit": False,            # commit manual SETELAH tugas selesai
    "max.poll.interval.ms": 1800000,
    "session.timeout.ms": 45000,
})
konsumen.subscribe([TOPIC_TUGAS], on_assign=lapor_siap)
print(f"Worker {HOST} menunggu tugas dari '{TOPIC_TUGAS}' ... (Interrupt untuk berhenti)")

terakhir_hidup = 0
try:
    while True:
        msg = konsumen.poll(1.0)
        if time.time() - terakhir_hidup > 10:
            kirim(TOPIC_STATUS, HOST, {"run": "-", "worker": HOST, "status": "HIDUP", "waktu": time.time()})
            terakhir_hidup = time.time()
        if msg is None:
            continue
        if msg.error():
            if msg.error().code() != KafkaError._PARTITION_EOF:
                print("Kafka:", msg.error())
            continue
        try:
            tugas = json.loads(msg.value())
        except ValueError:
            print("Pesan rusak dilewati, offset", msg.offset())
            konsumen.commit(msg, asynchronous=False)
            continue
        print(f"\nMenerima tugas {tugas['tugas']} [{tugas['jenis']}] partisi data {tugas['partisi_kafka']} "
              f"(partisi tugas {msg.partition()}, offset {msg.offset()})")
        kerjakan(tugas)
        try:
            konsumen.commit(msg, asynchronous=False)
        except KafkaException as e:
            print("Commit gagal (rebalance), lanjut:", e)    # aman: Master menimpa kiriman ganda
except KeyboardInterrupt:
    print("\nWorker dihentikan.")
finally:
    try:
        kirim(TOPIC_STATUS, HOST, {"run": "-", "worker": HOST, "status": "BERHENTI", "waktu": time.time()})
        produser.flush(10)
    finally:
        konsumen.close()

Worker NthaanSnn-w4 menunggu tugas dari 'wiki3-tugas' ... (Interrupt untuk berhenti)
Partisi tugas yang dipegang: [0]

Menerima tugas 0 [olah] partisi data 0 (partisi tugas 0, offset 8)
  ingest p0: 54,878 baris file, 9 kosong dibuang, 0.03 dtk
  W1_per_menit            5,310 baris
  W2_bot_wiki               287 baris
  W3_selisih_ringkas          2 baris
  W4_selisih_hist         5,233 baris
  W5_idwiki_halaman          75 baris
  W6_jendela_hist            66 baris
  W7_kandidat             2,546 baris
  valid 54,854 (harap 54,854) | rusak: kosong 9, corrupt 11, field_wajib 4 (harap total 24)
        id         wiki type                                                    title                  ts  length_old  length_new  selisih_byte
2651834377 wikidatawiki edit                                               Q100893590 2026-11-18 03:28:59       12171       12276           105
3535906029  commonswiki edit File:08.02 副總統出席「2026台灣盃火箭競賽決賽閉幕暨頒獎典禮」 - 55437021003.jpg 2026-11-18 03:28:58    